### 1.Import all necessary libraries

In [184]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

### 2.Load Data

In [185]:
merged_df = pd.read_csv(r"C:\Users\dhama\Desktop\MarketPulse_NLP\merged_stock_sentiment_final.csv")

### 3.Enforce Datetime format

In [186]:
merged_df["Date"] = pd.to_datetime(merged_df["Date"])

### 4.Sort Data Before Target Creation


In [187]:
merged_df = merged_df.sort_values(by=["Stock", "Date"]).reset_index(drop=True)

### 5.Create Target Variable

In [188]:
merged_df["Target"] = (
    merged_df.groupby("Stock")["Daily_Return"]
             .shift(-1)
             .gt(0)
             .astype(int)
)

In [189]:
merged_df.to_csv(r"C:\Users\dhama\Desktop\MarketPulse_NLP\merged_stock_sentiment_with_target.csv", index=False)
print("Success: Final CSV saved with Target column!")

Success: Final CSV saved with Target column!


### 6.Boundary Truncation & Type Enforcement

* **`dropna(subset=["Target"])`**: Shifting a time-series dataset forward by one interval naturally generates missing values (`NaN`) for the final trailing row of each stock ticker, as no future row exists. This line drops those terminal boundary rows to prevent Scikit-Learn estimators from crashing during model fitting.
* **`.astype(int)`**: Pandas automatically casts a column into floating-point decimals (`1.0`, `0.0`) to handle missing `NaN` values. Once the null rows are truncated, this line converts the target vector back into clean classification integers (`0` and `1`), which is the exact format required by our classification algorithms.


In [190]:
merged_df = merged_df.dropna(subset=["Target"])
merged_df["Target"] = merged_df["Target"].astype(int)

### 7.Temporal Grouping for Chronological Splitting

* **`sort_values(by=["Date", "Stock"])`**: Re-orders the entire dataset so that rows are grouped day-by-day across all 10 stocks simultaneously. This is a critical data prep step right before our time split. It ensures that when we apply our chronological date masks (Train: 2020–2023 | Test: 2024), we capture a fair slice of every single stock in both our training and testing windows, completely eliminating "stock blindness."


In [191]:
merged_df = merged_df.sort_values(by=["Date", "Stock"]).reset_index(drop=True)

In [192]:
merged_df = merged_df[merged_df["Date"] <= "2024-03-04"].reset_index(drop=True)


### 8.Chronological Data Split(2020-2023 vs 2024)

In [193]:
train_mask = merged_df["Date"] <= "2023-12-31"
test_mask = merged_df["Date"] >= "2024-01-01"

In [194]:
merged_df.to_csv(r"C:\Users\dhama\Desktop\MarketPulse_NLP\merged_stock_sentiment_with_target.csv", index=False)
print(f"File updated with {len(merged_df)} rows!")


File updated with 10191 rows!


### 9.Define Technical Features

In [195]:
features_1 = ["Volume", "Price_Range", "MA_7", "MA_30", "Volatility_7"]

### 10.Define X_train,X_test and y_train,y_test

In [196]:
X_train = merged_df.loc[train_mask, features_1]
y_train = merged_df.loc[train_mask, "Target"]

X_test = merged_df.loc[test_mask, features_1]
y_test = merged_df.loc[test_mask, "Target"]

print(f"Data verification: Train rows={X_train.shape} | Test rows={X_test.shape}\n")

Data verification: Train rows=(9761, 5) | Test rows=(430, 5)



### 11.Scaling(For Logistic Regression)

In [197]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 12.Logistic Regression

In [198]:
lr = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced'
)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print("--- FINAL LOGISTIC REGRESSION RESULTS ---")
print("Locked Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))


--- FINAL LOGISTIC REGRESSION RESULTS ---
Locked Accuracy: 0.5093023255813953
Confusion Matrix:
 [[162  50]
 [161  57]]


### 13.Random Forest

In [199]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    bootstrap=True,
    random_state=42,  
    n_jobs=1          
)
rf.fit(X_train_scaled, y_train) 
y_pred_rf = rf.predict(X_test_scaled)

print("--- FINAL RANDOM FOREST RESULTS ---")
print("Locked Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))


--- FINAL RANDOM FOREST RESULTS ---
Locked Accuracy: 0.4697674418604651
Confusion Matrix:
 [[110 102]
 [126  92]]


### Let us train the model by adding Sentiment Score as an feature in it



### 14.Logistic Regression

In [200]:
features_2 = ["Volume", "Price_Range", "MA_7", "MA_30", "Volatility_7", "sentiment_score"]

X_train_2 = merged_df.loc[train_mask, features_2]
X_test_2 = merged_df.loc[test_mask, features_2]


scaler_2 = StandardScaler()
X_train_2_scaled = scaler_2.fit_transform(X_train_2)
X_test_2_scaled = scaler_2.transform(X_test_2)


lr_2 = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_2.fit(X_train_2_scaled, y_train)
y_pred_lr_2 = lr_2.predict(X_test_2_scaled)

print("--- FINAL LOGISTIC REGRESSION RESULTS ---")
print("Locked Accuracy:", accuracy_score(y_test, y_pred_lr_2))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr_2))


--- FINAL LOGISTIC REGRESSION RESULTS ---
Locked Accuracy: 0.47674418604651164
Confusion Matrix:
 [[105 107]
 [118 100]]


### 15.Random Forest

In [201]:
rf_2 = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", bootstrap=True, random_state=42, n_jobs=1)
rf_2.fit(X_train_2_scaled, y_train)
y_pred_rf_2 = rf_2.predict(X_test_2_scaled)

print("--- FINAL RANDOM FOREST RESULTS ---")
print("Locked Accuracy:", accuracy_score(y_test, y_pred_rf_2))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf_2))

--- FINAL RANDOM FOREST RESULTS ---
Locked Accuracy: 0.4604651162790698
Confusion Matrix:
 [[126  86]
 [146  72]]


In [202]:
import pickle

# Save the trained Random Forest model (Experiment 2: Technical + Sentiment)
model_save_path = r"C:\Users\dhama\Desktop\MarketPulse NLP\rf_sentiment_model.pkl"
with open(model_save_path, "wb") as f:
    pickle.dump(rf_2, f)

print("Success: Pre-trained Random Forest model saved as 'rf_sentiment_model.pkl'!")


Success: Pre-trained Random Forest model saved as 'rf_sentiment_model.pkl'!
